In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

PROJECT_ROOT = Path("/mnt/d/IDC409_Project15")
DB_PATH      = PROJECT_ROOT / "data" / "processed" / "house_prices.db"
FIGURES_DIR  = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

engine = create_engine(f"sqlite:///{DB_PATH}")

# Load from the database we built in Phase 1 — not from the raw CSV
df = pd.read_sql("SELECT * FROM house_prices_raw", con=engine)

print(f"Loaded from SQLite: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

Loaded from SQLite: 250,000 rows × 23 columns


,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.76,0.10,1990,...,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.52,0.08,2008,...,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.79,0.05,1997,...,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move


In [2]:
cols_to_drop = [
    "ID",           # row number, carries zero information
    "Floor_No",     # 46.5% of values are impossible (floor > total floors)
    "Total_Floors", # meaningless without a valid Floor_No
    "Year_Built",   # Age_of_Property already captures this
]

df = df.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns: {cols_to_drop}")
print(f"Remaining: {df.shape[1]} columns")

Dropped 4 columns: ['ID', 'Floor_No', 'Total_Floors', 'Year_Built']
Remaining: 19 columns


In [3]:
# Amenities looks like: "Playground, Gym, Garden, Pool, Clubhouse"
# First let's see every unique amenity that appears across all rows

all_amenities = set()
df['Amenities'].str.split(',').apply(
    lambda lst: all_amenities.update([x.strip() for x in lst])
)

print(f"All unique amenities found ({len(all_amenities)} total):")
for a in sorted(all_amenities):
    count = df['Amenities'].str.contains(a, case=False).sum()
    print(f"  {a:15s} → {count:,} properties ({count/len(df)*100:.1f}%)")

All unique amenities found (5 total):
  Clubhouse       → 150,059 properties (60.0%)
  Garden          → 150,177 properties (60.1%)
  Gym             → 149,960 properties (60.0%)
  Playground      → 149,780 properties (59.9%)
  Pool            → 150,096 properties (60.0%)


In [4]:
# Replace the text column with useful numbers

# 1. Total count of amenities per property
df['Amenities_Count'] = df['Amenities'].str.split(',').apply(len)

# 2. Binary flag for each common amenity (1 = has it, 0 = doesn't)
common_amenities = ['Gym', 'Pool', 'Garden', 'Playground', 'Clubhouse']

for amenity in common_amenities:
    col_name = f"Has_{amenity}"
    df[col_name] = df['Amenities'].str.contains(amenity, case=False, na=False).astype(int)

# 3. Drop the original text column — we've extracted everything useful
df = df.drop(columns=['Amenities'])

print("Amenities text column replaced with these numeric features:")
new_cols = ['Amenities_Count'] + [f'Has_{a}' for a in common_amenities]
for col in new_cols:
    print(f"  {col:20s} → mean = {df[col].mean():.2f}")

Amenities text column replaced with these numeric features:
  Amenities_Count      → mean = 3.00
  Has_Gym              → mean = 0.60
  Has_Pool             → mean = 0.60
  Has_Garden           → mean = 0.60
  Has_Playground       → mean = 0.60
  Has_Clubhouse        → mean = 0.60


In [5]:
# Price_per_SqFt = Price_in_Lakhs / Size_in_SqFt
# PROBLEM: to predict Price_in_Lakhs, you can't use a column derived FROM it
# This is called "data leakage" — the model would look perfect but be useless

# Verify the column is indeed derived from the target
df['_check'] = (df['Price_in_Lakhs'] / df['Size_in_SqFt']).round(4)
match_pct = (df['Price_per_SqFt'].round(4) == df['_check']).mean() * 100
df = df.drop(columns=['_check'])

print(f"Price_per_SqFt matches Price/Size calculation: {match_pct:.1f}% of rows")
print()
print("DECISION: Keep Price_per_SqFt in the dataframe for EDA visualisations.")
print("          It will be EXCLUDED when we build models in Phase 5.")
print("          (Using it would be data leakage — cheating the model.)")

Price_per_SqFt matches Price/Size calculation: 1.0% of rows

DECISION: Keep Price_per_SqFt in the dataframe for EDA visualisations.
          It will be EXCLUDED when we build models in Phase 5.
          (Using it would be data leakage — cheating the model.)


In [6]:
numeric_cols = [
    'Size_in_SqFt', 'BHK', 'Age_of_Property',
    'Nearby_Schools', 'Nearby_Hospitals', 'Amenities_Count', 'Price_in_Lakhs'
]

print(f"{'Column':<25} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8}")
print("-" * 65)
for col in numeric_cols:
    print(f"{col:<25} {df[col].min():>8.1f} {df[col].max():>8.1f} "
          f"{df[col].mean():>8.1f} {df[col].std():>8.1f}")

Column                         Min      Max     Mean      Std
-----------------------------------------------------------------
Size_in_SqFt                 500.0   5000.0   2749.8   1300.6
BHK                            1.0      5.0      3.0      1.4
Age_of_Property                2.0     35.0     18.5      9.8
Nearby_Schools                 1.0     10.0      5.5      2.9
Nearby_Hospitals               1.0     10.0      5.5      2.9
Amenities_Count                1.0      5.0      3.0      1.4
Price_in_Lakhs                10.0    500.0    254.6    141.3


In [7]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f"Categorical columns: {len(cat_cols)}\n")
for col in cat_cols:
    n = df[col].nunique()
    top = df[col].value_counts().index[0]
    print(f"  {col:<35} {n:>4} unique values   (most common: {top})")

Categorical columns: 11

  State                                 20 unique values   (most common: Odisha)


/tmp/ipykernel_2971/1614360390.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


  City                                  42 unique values   (most common: Coimbatore)
  Locality                             500 unique values   (most common: Locality_296)
  Property_Type                          3 unique values   (most common: Villa)
  Furnished_Status                       3 unique values   (most common: Unfurnished)
  Public_Transport_Accessibility         3 unique values   (most common: High)
  Parking_Space                          2 unique values   (most common: No)
  Security                               2 unique values   (most common: Yes)
  Facing                                 4 unique values   (most common: West)
  Owner_Type                             3 unique values   (most common: Broker)
  Availability_Status                    2 unique values   (most common: Under_Construction)


In [8]:
locality_n = df['Locality'].nunique()
city_n     = df['City'].nunique()
state_n    = df['State'].nunique()

print(f"Location columns unique value counts:")
print(f"  State    : {state_n}  ← manageable")
print(f"  City     : {city_n}  ← manageable")
print(f"  Locality : {locality_n}  ← too many for one-hot encoding")
print()
print("DECISION: Locality will be DROPPED in Phase 4 (Feature Engineering).")
print("          State + City together capture location well enough.")

Location columns unique value counts:
  State    : 20  ← manageable
  City     : 42  ← manageable
  Locality : 500  ← too many for one-hot encoding

DECISION: Locality will be DROPPED in Phase 4 (Feature Engineering).
          State + City together capture location well enough.


In [9]:
print(f"Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n{'Column':<35} {'Dtype':<12} {'Non-null':>10}")
print("-" * 60)
for col in df.columns:
    dtype   = str(df[col].dtype)
    nonnull = df[col].notnull().sum()
    print(f"  {col:<33} {dtype:<12} {nonnull:>10,}")

Clean dataset: 250,000 rows × 24 columns

Column                              Dtype          Non-null
------------------------------------------------------------
  State                             str             250,000
  City                              str             250,000
  Locality                          str             250,000
  Property_Type                     str             250,000
  BHK                               int64           250,000
  Size_in_SqFt                      int64           250,000
  Price_in_Lakhs                    float64         250,000
  Price_per_SqFt                    float64         250,000
  Furnished_Status                  str             250,000
  Age_of_Property                   int64           250,000
  Nearby_Schools                    int64           250,000
  Nearby_Hospitals                  int64           250,000
  Public_Transport_Accessibility    str             250,000
  Parking_Space                     str             250,0

In [10]:
# Save to SQLite as a new table
df.to_sql("house_prices_clean", con=engine, if_exists="replace", index=False)
print("Saved to SQLite: table 'house_prices_clean'")

# Also save as CSV (easy to share with teammates)
clean_csv = PROJECT_ROOT / "data" / "processed" / "house_prices_clean.csv"
df.to_csv(clean_csv, index=False)
print(f"Saved to CSV   : {clean_csv}")

print(f"\nFinal shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Saved to SQLite: table 'house_prices_clean'
Saved to CSV   : /mnt/d/IDC409_Project15/data/processed/house_prices_clean.csv

Final shape: 250,000 rows × 24 columns
